# Flat-field calibration from amorphous scattering

This notebook turns the pyFAI tutorial **"Flat-field calibration from amorphous scattering"**
(<https://www.silx.org/doc/pyFAI/dev/usage/tutorial/Flatfield.html>) into a reusable calibration
run. The method is from Weng *et al.*, *J. Synchrotron Rad.* **30**, 546–554 (2023): instead of
illuminating the detector evenly, which a small beam cannot do, we measure an amorphous scatterer
whose signal depends only on the scattering angle. After geometry and polarization corrections,
whatever still varies around each ring is the detector's own pixel-to-pixel response.

Measuring at several detector (or sample) positions moves the scattering pattern, beam-stop
shadow and gaps across the detector. A pixel-wise median of the per-position maps then removes
position-dependent artefacts and fills the blind regions.

**Two data sources**

* `DATA_SOURCE = "tutorial"`: the ESRF ID31 example (Pilatus 2M CdTe, 75 keV, 3×3 detector
  positions, silver behenate calibrant, amorphous carbon scatterer). The data are downloaded
  automatically.
* `DATA_SOURCE = "maxima"`: MAXIMA EIGER2 CdTe 1M `*_master.h5` files. Each position needs an
  amorphous-scattering exposure, plus either a calibrant exposure or an existing `.poni`.
  The DECTRIS `pixel_mask` and the 2³²−1 flag value are masked automatically.

The result is saved as a versioned bundle (`.edf` for pyFAI, `.h5`, `.json`). Optionally it is
published to **data.htmdec.org** as an immutable version, and can be promoted to "current" for
workflows to use.

> **Note for MAXIMA:** the EIGER2 already applies DECTRIS's factory flatfield on the detector, so
> the map produced here is a *residual* correction applied on top of it. Apply it in pyFAI with
> `flat=` together with the detector `pixel_mask`.

## 0. Parameters

In [ ]:
# ---- choose the data -------------------------------------------------------
DATA_SOURCE = "tutorial"          # "tutorial" or "maxima"

# ---- common settings -------------------------------------------------------
NPT = 512                          # radial bins for the median profile
MIN_VALID = 5                      # a pixel needs at least this many positions to be kept
MAX_RINGS = 4                      # calibrant rings used for geometry refinement
OUTPUT_DIR = "output"
NOTES = ""                         # free text stored with the calibration record

# ---- centre-position geometry ------------------------------------------------
# Default: load the centre geometry from a PONI file ("initial_poni" below) and refine it
# automatically on the calibrant rings. Set INTERACTIVE_CALIBRATION = True to pick the rings
# by hand in the widget instead (see the instructions in section 6).
INTERACTIVE_CALIBRATION = False
REFINE_INITIAL = True              # refine the initial PONI on the centre calibrant image

# ---- tutorial (ESRF ID31) ---------------------------------------------------
TUTORIAL = dict(
    energy_keV=75,
    polarization=0.999,            # synchrotron: horizontally polarized
    detector="Pilatus2M_CdTe",
    calibrant="AgBh",
    center=5,
    # centre geometry from the tutorial's published result (file next to this notebook)
    initial_poni="tutorial_center.poni",
    # position: (calibration scan, amorphous scan)
    scans={1: (1, 5), 2: (7, 6), 3: (8, 9), 4: (11, 12), 5: (14, 13),
           6: (15, 16), 7: (18, 17), 8: (19, 20), 9: (22, 21)},
    # 3x3 display layout matching the stage translation
    layout={1: (0, 2), 2: (0, 1), 3: (0, 0), 4: (1, 0), 5: (1, 1),
            6: (1, 2), 7: (2, 2), 8: (2, 1), 9: (2, 0)},
    fix_rotations=True,            # detector assumed perpendicular: tilt reset, rot1/rot2 fixed
)

# ---- MAXIMA (EIGER2 CdTe 1M) -------------------------------------------------
MAXIMA = dict(
    polarization=0.0,              # lab source: unpolarized (pyFAI: 0 = circular/unpolarized)
    detector="Eiger2CdTe_1M",
    calibrant="alpha_Al2O3",       # confirm which calibrant is mounted on the station
    center=0,
    initial_poni=None,             # None: use the centre position's own "poni" below
    fix_rotations=False,           # MAXIMA detector is tilted; keep refined rotations
    # One entry per position. "scattering" is required. Give "poni" (e.g. from the Dagster
    # pipeline) or "calibration" (+ coordinates in mm relative to the centre) for refinement.
    positions=[
        # dict(position=0, scattering=".../amorphous_p0_master.h5",
        #      calibration=".../XRD_calibrant_master.h5", poni=".../XRD_calibrant.poni",
        #      coordinates=(0, 0, 0)),
    ],
)

# ---- publishing to data.htmdec.org ------------------------------------------
PUBLISH = False                    # upload this calibration as a new immutable version
PROMOTE = False                    # also make it the "current" flat-field for this detector
# GIRDER_API_URL, GIRDER_API_KEY and FLATFIELD_FOLDER_ID are read from the environment / .env

## 1. Imports and set-up

### Note that there will be an error about not being able to import PyOpenCl which you can ignore. That is only needed if you want pyFAI to use a GPU.

In [ ]:
%matplotlib inline
import copy
import time
import sys
from pathlib import Path

import numpy
import h5py
import hdf5plugin  # noqa: F401  registers bitshuffle/LZ4 filters
from matplotlib.pyplot import subplots
import fabio
from silx.resources import ExternalResources
import pyFAI
from pyFAI.gui import jupyter

import flat_field
from flat_field import Position, maxima_position, build_mask, position_flat, merge_flats
from flat_field import to_pyfai_flat, flat_stats, robust_spread, pixel_scale_spread, save_flatfield
from flat_field.calibration import install_extract_cpt, predict_geometry, refine_position
from flat_field.io import build_metadata, make_version

install_extract_cpt()  # the tutorial's control-point extraction (honours mask, max_rings)
print(f"Running pyFAI version {pyFAI.version}, flat_field {flat_field.__version__} on python {sys.version}")
t0 = time.perf_counter()

In [ ]:
if DATA_SOURCE == "tutorial":
    P = TUTORIAL
    polarization = P["polarization"]
    energy = P["energy_keV"]
    wavelength = 1e-10 * pyFAI.units.hc / energy
elif DATA_SOURCE == "maxima":
    P = MAXIMA
    if not P["positions"]:
        raise ValueError("Fill MAXIMA['positions'] in the parameters cell.")
    polarization = P["polarization"]
    wavelength = None  # taken from the data below
else:
    raise ValueError(DATA_SOURCE)

npt = NPT
detector = pyFAI.detector_factory(P["detector"])
calibrant = pyFAI.calibrant.CALIBRANT_FACTORY(P["calibrant"])

## 2. Data

For the tutorial the nine positions are downloaded from silx.org as one archive (about 67 MB).
For MAXIMA each position is read from its `*_master.h5` file. The matching `*_data_000001.h5`
must sit next to it.

In [ ]:
if DATA_SOURCE == "tutorial":
    downloader = ExternalResources("flatfield", "http://www.silx.org/pub/pyFAI/testimages")
    all_files = downloader.getdir("flatfield_ID31.tar.bz2")
    master_file = [i for i in all_files if i.endswith("calibration_0001.h5")][0]
    print(master_file)

## 3. Organising the data

Each `Position` holds the calibrant image, the amorphous image and the stage coordinates. The
geometry, control points and flat-field are added as we go.

In [ ]:
if DATA_SOURCE == "tutorial":
    data = {pos: Position.init(master_file, pos, cal, sca) for pos, (cal, sca) in P["scans"].items()}
    layout = P["layout"]
else:
    data = {}
    for spec in P["positions"]:
        p = maxima_position(spec["position"], spec["scattering"], spec.get("calibration"),
                            spec.get("coordinates", (0.0, 0.0, 0.0)), spec.get("poni"))
        data[p.position] = p
    n = len(data)
    ncol = int(numpy.ceil(numpy.sqrt(n)))
    layout = {pos: divmod(i, ncol) for i, pos in enumerate(data)}
    first = next(iter(data.values()))
    ref_poni = P["initial_poni"] or data[P["center"]].poni
    if ref_poni:  # use the detector definition (incl. orientation) stored in the PONI
        detector = pyFAI.load(ref_poni).detector
    energy = first.meta["photon_energy_eV"] / 1000
    wavelength = 1e-10 * pyFAI.units.hc / energy
    print("Detector", first.meta.get("description"), first.meta.get("detector_number"),
          f"| photon energy {energy:.3f} keV | threshold {first.meta.get('threshold_energy_eV')} eV")

calibrant.wavelength = wavelength
center = data[P["center"]]
center

## 4. Masking invalid pixels

Following the tutorial, every pixel that is negative in any image is masked, together with the
detector's static mask (module gaps). For MAXIMA the DECTRIS `pixel_mask` and the flagged pixels
are added as well.

In [ ]:
images = []
for p in data.values():
    images += [p.scattering_data, p.calibration_data]
mask = build_mask(detector, images, extra_masks=[p.pixel_mask for p in data.values()])
detector.mask = mask
print(f"{int(mask.sum())} masked pixels ({mask.mean():.2%})")
fig, ax = subplots()
ax.imshow(detector.mask);

## 5. Overview of the raw data

In [ ]:
def show_grid(images, layout, extra=None, size=12):
    nrow = max(r for r, c in layout.values()) + 1
    ncol = max(c for r, c in layout.values()) + 1
    fig, ax = subplots(nrow, ncol, figsize=(size, size * nrow / ncol), squeeze=False)
    for pos, img in images.items():
        if img is None:
            continue
        r, c = layout[pos]
        kw = (extra or {}).get(pos, {})
        jupyter.display(img, ax=ax[r, c], **kw)
        ax[r, c].set_title(f"position {pos}")
    return ax

if any(p.calibration_data is not None for p in data.values()):
    show_grid({k: p.calibration_data for k, p in data.items()}, layout);

In [ ]:
show_grid({k: p.scattering_data for k, p in data.items()}, layout);

## 6. Calibration of the central position

**Default (no clicking).** The centre geometry is loaded from the PONI file given as
`initial_poni` (for the tutorial, `tutorial_center.poni` next to this notebook). It is then
refined automatically on the first `MAX_RINGS` calibrant rings. For MAXIMA positions that
already have their own `.poni`, this step only loads it.

**Interactive (`INTERACTIVE_CALIBRATION = True`, or no PONI available).** You pick the rings by
hand in the widget, as in the original tutorial:

1. Check the figure is interactive: it has a small toolbar and shows x/y coordinates under the
   mouse. If it is a plain picture, start JupyterLab from the project environment
   (`uv run jupyter lab`) and re-run the cells.
2. Type the ring number in the **Ring#** box under the image *before* clicking that ring.
   0 is the innermost ring of the calibrant, 1 the next, and so on.
3. Click once on that ring with a **true right-click**: on a Mac, a **two-finger click/tap** on
   the trackpad (turn on "Secondary click" in System Settings → Trackpad if needed), or the right
   button of a mouse. **Ctrl-click does not work**: the widget receives it as a left click with
   Control held and ignores it (the cell output shows "Unknown combination"). Alternatively, hold
   the **N** key and left-click (click once in the figure first so it receives key presses).
   pyFAI picks a whole group of points along the ring.
4. Repeat for 3–4 rings, changing **Ring#** before each. To delete a wrong group, hold **D** and
   left-click on it. Other keys + left-click: **V** adds the nearest peak to the current group,
   **B** starts a new single-point group, **M** adds more points.
5. Click the **Refine** button under the image. This fits a first geometry to your points.
6. Run the next cell and press **Enter once** in its input box. The notebook then resets the
   tilt (if `fix_rotations`), extracts control points automatically and refines.

You don't press Enter after each click.

With `fix_rotations=True` (tutorial) the detector is assumed perpendicular to the beam: the tilt
is reset and `rot1`/`rot2` are fixed.

In [ ]:
initial_poni = P["initial_poni"]
if initial_poni and not Path(initial_poni).exists():
    raise FileNotFoundError(f"initial_poni {initial_poni!r} not found (paths are relative to {Path.cwd()})")
INTERACTIVE = INTERACTIVE_CALIBRATION or (initial_poni is None and center.poni is None)
print("Interactive ring picking" if INTERACTIVE else f"Using PONI: {initial_poni or center.poni}")
if INTERACTIVE:
    %matplotlib widget
    from pyFAI.gui.jupyter.calib import Calibration
    extra_mask = center.calibration_data < 5000   # tutorial: keep only the brightest rings
    calib = Calibration(center.calibration_data,
                        calibrant=calibrant,
                        wavelength=calibrant.wavelength,
                        detector=copy.deepcopy(detector),
                        mask=extra_mask)  # the mask option mangles the detector's mask!

In [ ]:
if INTERACTIVE:
    input("Pick the rings in the widget above and click 'Refine', then press Enter here ...")
    %matplotlib inline
    calib.mask = None
    calib.geoRef.detector = detector
    if P["fix_rotations"]:
        f2d = calib.geoRef.getFit2D()
        f2d["tilt"] = 0
        calib.geoRef.setFit2D(**f2d)
        calib.fixed += ["rot1", "rot2"]
    calib.extract_cpt(max_rings=MAX_RINGS)
    calib.refine()
    ai = pyFAI.load(calib.geoRef)
    center.control_points = calib.peakPicker.points
else:
    start = pyFAI.load(initial_poni or center.poni)
    start.detector = detector
    if REFINE_INITIAL and center.calibration_data is not None and (initial_poni or not center.poni):
        if P["fix_rotations"]:
            f2d = start.getFit2D(); f2d["tilt"] = 0; start.setFit2D(**f2d)
        fixed = ("rot1", "rot2") if P["fix_rotations"] else ()
        ai, center.control_points = refine_position(center.calibration_data, detector.mask, detector,
                                                    calibrant, wavelength, start, MAX_RINGS, fixed)
    else:
        ai = start
ai = ai.promote("pyFAI.integrator.azimuthal.AzimuthalIntegrator")
ai.detector = detector
center.ai = ai
print(ai)

### From amorphous scattering to a flat-field (centre position)

The amorphous image is averaged azimuthally in three ways: a plain mean, sigma-clipping and a
median. The median (`medfilt1d_ng`) is the most robust against hot or masked pixels and is the
one used. `guess_polarization` checks the polarization factor. For a synchrotron it should be
close to 1. For a lab source, 0 (unpolarized) is the expected value.

In [ ]:
it = ai.integrate1d(center.scattering_data, npt, polarization_factor=polarization, error_model="no",
                    method=("no", "csr", "cython"))
sc = ai.sigma_clip(center.scattering_data, npt, polarization_factor=polarization, error_model="azimuthal",
                   method=("no", "csr", "cython"), thres=0, max_iter=3)
md_ = ai.medfilt1d_ng(center.scattering_data, npt, polarization_factor=polarization,
                      method=("full", "csr", "cython"))
fig, ax = subplots()
ax = jupyter.plot1d(it, label="integrate", ax=ax)
ax.errorbar(*sc, alpha=0.7, label="sigma-clip")
ax.plot(*md_, alpha=0.7, label="median")
ax.legend();

In [ ]:
try:
    print(f"Approximate polarization factor: "
          f"{ai.guess_polarization(numpy.nan_to_num(center.scattering_data), unit='q_nm^-1', target_rad=10):.4f}")
except Exception as exc:  # depends on q-range covered
    print("guess_polarization failed:", exc)

The median profile is projected back onto the detector (`calcfrom1d`). The pixel-wise ratio of
this ideal image to the measured one is the response (gain) map. Masked and empty pixels are NaN.

In [ ]:
flat, _ = position_flat(ai, center.scattering_data, detector.mask, npt, polarization)
ai.reset()
ax = jupyter.display(flat)
ax.set_title("gain map, centre position");

## 7. Geometry of the other positions

Each position's geometry is predicted from the centre by adding the stage displacement (mm → m)
to `poni1`/`poni2`, then refined on its own calibrant image. Positions that already have a `.poni`
(MAXIMA) use it directly.

In [ ]:
for pos, p in data.items():
    if p is center:
        continue
    if p.poni:
        p.ai = pyFAI.load(p.poni).promote("pyFAI.integrator.azimuthal.AzimuthalIntegrator")
        p.ai.detector = detector
        print(f"#### Position {pos}: loaded {p.poni}")
        continue
    if p.calibration_data is None:
        raise ValueError(f"Position {pos} needs either a poni or a calibration image")
    ain = predict_geometry(ai, center.coordinates, p.coordinates)
    fixed = ("rot1", "rot2") if P["fix_rotations"] else ()
    p.ai, p.control_points = refine_position(p.calibration_data, detector.mask, detector,
                                             calibrant, wavelength, ain, MAX_RINGS, fixed)
    p.ai = p.ai.promote("pyFAI.integrator.azimuthal.AzimuthalIntegrator")
    p.ai.detector = detector
    print(f"#### Position {pos} ####")
    print(p.ai)

In [ ]:
if any(p.calibration_data is not None for p in data.values()):
    show_grid({k: p.calibration_data for k, p in data.items()}, layout,
              extra={k: dict(ai=p.ai, cp=p.control_points) for k, p in data.items()
                     if p.control_points is not None});

## 8. Flat-field of each position

In [ ]:
for p in data.values():
    p.flat, _ = position_flat(p.ai, p.scattering_data, detector.mask, npt, polarization)
    p.ai.reset()  # free the integration engine (each one holds a large sparse matrix)
show_grid({k: p.flat for k, p in data.items()}, layout);

## 9. Merging the maps

A pixel-wise median over all positions removes position-dependent artefacts and fills blind
regions. Pixels measured at fewer than `MIN_VALID` positions are left undetermined (NaN).

In [ ]:
gain, n_valid = merge_flats([p.flat for p in data.values()], min_valid=MIN_VALID)
ax = jupyter.display(gain)
cb = ax.figure.colorbar(ax.images[0])
pos = numpy.linspace(0.8, 1.2, 5)
cb.set_ticks(pos, labels=[str(i) for i in pos]);
print(f"{numpy.isfinite(gain).mean():.2%} of pixels determined")

### Quality check

Apply the merged flat to each amorphous image and recompute the residual map. If the
correction works, the spread of the residual drops towards the counting-noise level
(about 1/√counts per pixel). Two numbers are reported, each as half the 16–84 % range so
a few hot pixels don't dominate:

* **pixel-scale**: the map divided by its local 7×7 median. This is the detector response the
  flat-field is meant to remove, so it should drop clearly.
* **total**: also includes smooth structure that is specific to one position (parasitic
  scatter, absorption, shadows). The median merge rejects that on purpose, so it may change
  little.

In [ ]:
flat_pyfai = to_pyfai_flat(gain)   # pyFAI divides by the flat
qc = {}
for k, p in data.items():
    corrected = p.scattering_data / flat_pyfai
    resid, _ = position_flat(p.ai, corrected, detector.mask, npt, polarization)
    p.ai.reset()
    qc[k] = dict(total_before=robust_spread(p.flat), total_after=robust_spread(resid),
                 pixel_before=pixel_scale_spread(p.flat), pixel_after=pixel_scale_spread(resid))
    print(f"position {k}: pixel-scale {qc[k]['pixel_before']:.4f} -> {qc[k]['pixel_after']:.4f}"
          f"   | total {qc[k]['total_before']:.4f} -> {qc[k]['total_after']:.4f}")

## 10. Saving the result

The map is a gain (ideal / measured). pyFAI *divides* by the flat-field, so the reciprocal is
saved, as in the tutorial. The calibration is written as a versioned bundle together with its
metadata (detector, energy, threshold, inputs, parameters, statistics, software versions).

In [ ]:
version = make_version()
first = next(iter(data.values()))
if DATA_SOURCE == "maxima":
    det_id = first.meta.get("detector_number") or P["detector"]
    det_info = {k: first.meta.get(k) for k in (
        "description", "detector_number", "sensor_material", "sensor_thickness_m",
        "photon_energy_eV", "element", "threshold_energy_eV", "firmware",
        "flatfield_correction_applied", "countrate_correction_applied",
        "pixel_mask_applied", "virtual_pixel_correction_applied")}
    det_info["applies_on_top_of"] = "DECTRIS factory flatfield applied on the detector"
else:
    det_id = P["detector"]
    det_info = {"description": P["detector"], "photon_energy_eV": energy * 1000}
det_info["pyfai_detector"] = P["detector"]

inputs = [{"position": str(k), "scattering": str(p.scattering_idx), "calibration": str(p.calibration_idx),
           "poni": p.poni, "coordinates": list(map(float, p.coordinates)),
           "geometry": {kk: float(getattr(p.ai, kk)) for kk in ("dist", "poni1", "poni2", "rot1", "rot2", "rot3")},
           "source": p.meta.get("source"), "collected": p.meta.get("data_collection_date"),
           "qc": qc[k]} for k, p in data.items()]
parameters = {"data_source": DATA_SOURCE, "npt": npt, "min_valid": MIN_VALID, "max_rings": MAX_RINGS,
              "polarization_factor": polarization, "calibrant": P["calibrant"], "energy_keV": energy,
              "wavelength_m": wavelength, "fix_rotations": P["fix_rotations"], "n_positions": len(data)}
stats = flat_stats(flat_pyfai)
metadata = build_metadata(version=version, detector=det_info, parameters=parameters,
                          inputs=inputs, stats=stats, notes=NOTES)
basename = f"flatfield_{det_id}_{energy:.2f}keV_{version}".replace(" ", "_")
saved = save_flatfield(OUTPUT_DIR, flat_pyfai, gain, n_valid, detector.mask, metadata, basename)
for k in ("edf", "h5", "json"):
    print(saved[k])
stats

## 11. Publishing to data.htmdec.org (optional)

With `PUBLISH = True` the bundle is uploaded as a **new, immutable item** in the folder given by
`FLATFIELD_FOLDER_ID`, with the metadata record attached (`data_type = "xrd_flatfield"`). With
`PROMOTE = True` the folder's `current_flatfield` pointer for this detector is moved to the new
version. Workflows that ask for the "current" flat-field then pick it up, and every earlier
version stays available by its version label.

Credentials come from the environment or a `.env` file (`GIRDER_API_URL`, `GIRDER_API_KEY`,
`FLATFIELD_FOLDER_ID`). Never put the key in the notebook.

In [ ]:
from flat_field import portal

if PUBLISH:
    cfg = portal.PortalConfig.from_env()
    if not (cfg.api_key and cfg.folder_id):
        raise RuntimeError("Set GIRDER_API_KEY and FLATFIELD_FOLDER_ID (environment or .env)")
    gc = portal.connect(cfg)
    result = portal.publish_flatfield(gc, cfg.folder_id, [saved["edf"], saved["h5"], saved["json"]],
                                      saved["record"], detector_id=str(det_id), energy_keV=energy,
                                      version=version, promote=PROMOTE, api_url=cfg.api_url)
    print(result)
else:
    print("PUBLISH is False: calibration saved locally only.")

### Using a published flat-field in a workflow

```python
from flat_field import portal
import fabio
cfg = portal.PortalConfig.from_env()
gc = portal.connect(cfg)
got = portal.fetch_flatfield(gc, cfg.folder_id, "E-02-0375", "flatfields/", version="current")
edf = next(p for n, p in got["paths"].items() if n.endswith(".edf"))
flat = fabio.open(str(edf)).data
ai.integrate1d(img, 10000, unit="q_nm^-1", flat=flat, mask=pixel_mask != 0)
```

In [ ]:
print(f"Total run time: {time.perf_counter() - t0:.3f}s.")